# RG and MLE with a binary LLM judge

Independent calibration and unlabeled samples; compare with PPI++. The fixed synthetic judge has sensitivity 0.8 and specificity 0.85. RG uses the official finite-sample adjustment and logit interval; MLE uses observed Fisher information and a logit interval. PPI++ uses its usual Wald interval.


In [1]:
import numpy as np
from openppi import Mean, PPIData, get_method

rng = np.random.default_rng(42)
y = rng.binomial(1, .4, 3000)
judge = rng.binomial(1, np.where(y == 1, .8, .15))
data = PPIData(y[:400], judge[:400], judge[400:])


In [2]:
for name in ('RG', 'MLE', 'PPI++'):
    result = get_method(name).inference(data, target=Mean(), alpha=.05)
    print(f'{name}: estimate={result.estimate:.4f}, SE={result.standard_error:.4f}, CI={result.confidence_interval}')
    if name != 'PPI++':
        print('  Youden index:', result.diagnostics['youden_index'])


RG: estimate=0.3962, SE=0.0323, CI=(0.33493313076813025, 0.4608847139781864)
  Youden index: 0.6465987444684573
MLE: estimate=0.3924, SE=0.0194, CI=(0.3551153097133707, 0.43097310457203736)
  Youden index: 0.6539609366370058
PPI++: estimate=0.3924, SE=0.0194, CI=(0.3543980556656493, 0.4303866437531475)


RG requires both true label classes. MLE requires all four calibration cells to be positive and positive judge orientation. A near-zero adjusted Youden index makes RG unstable and raises below its configured tolerance. Predictions must be 0/1, not soft scores.
